# 7.2b Trees in Python (scikit-learn)

**Machine learning series, a companion to 7.2.** The same data, the same question, in **Python**, with **scikit-learn**: the most widely used machine learning library. If you work in Python, this is how 7.1 and 7.2 look in your language. If you work in R, it shows that the ideas, not the package, are what matter.

| | |
|---|---|
| ⏱️ **Run time** | about 5 minutes |
| 💰 **Data cost** | **\$0**: it reads `ml_t2d_2yr.csv`, saved by **7.1** |
| 📋 **Kernel** | **Python 3** (not R) |

| tidymodels (R, 7.1 and 7.2) | scikit-learn (Python, here) |
|---|---|
| `initial_split(strata = )` | `train_test_split(stratify = )` |
| `recipe()` with `step_dummy()`, `step_normalize()` | `ColumnTransformer` with `OneHotEncoder`, `StandardScaler` |
| `workflow()` | `make_pipeline()` |
| `logistic_reg()` | `LogisticRegression()` |
| `boost_tree()` with xgboost | `HistGradientBoostingClassifier()` |
| `vfold_cv()` + `tune_grid()` | `GridSearchCV()` |
| `roc_auc_vec()` | `roc_auc_score()` |
| permutation importance, by hand | `permutation_importance()` |

> ⚠️ **Keep All of Us data in the Workbench.** Share only summaries, with no counts of 1 to 20. Clear outputs before committing.

## 1. Setup and the data

`aou_setup.py` finds your bucket, as `aou_setup.R` does for R. scikit-learn is usually installed in the app; if not, the cell installs it.

In [ ]:
import sys
sys.path.insert(0, "..")
from aou_setup import *          # DATA_DIR, hide_small_counts()

import importlib.util
if importlib.util.find_spec("sklearn") is None:
    %pip install --quiet scikit-learn

import os
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.inspection import permutation_importance

print("scikit-learn", sklearn.__version__)

ml = pd.read_csv(os.path.join(DATA_DIR, "ml_t2d_2yr.csv"), dtype={"person_id": str})
ml["income"] = ml["income"].str.replace("$", "", regex=False)   # files saved before November 2026 had $ signs

NUMERIC     = ["age", "bmi", "sbp"]
CATEGORICAL = ["sex_at_birth", "general_health", "income", "education", "smoking_status"]

X = ml[NUMERIC + CATEGORICAL]
y = (ml["t2d_future"] == "yes").astype(int)      # 1 = diagnosed within 2 years

pd.DataFrame([{"people": len(ml), "new_t2d": int(y.sum()), "pct_new_t2d": round(100 * y.mean(), 2)}])

**🔍 Read the output:** the scikit-learn version, then the same people and new cases as in 7.1.

## 2. Training and test sets

`train_test_split()` with `stratify=y` keeps the same share of new cases in both sets; `random_state` plays the part of R's `set.seed()`. Python's random numbers differ from R's, so this is a **different** 75/25 split from 7.1's: expect AUCs close to, but not exactly, those in 7.1 and 7.2.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=2026)

pd.DataFrame({"people": [len(y_train), len(y_test)],
              "pct_new_t2d": [round(100 * y_train.mean(), 2), round(100 * y_test.mean(), 2)]},
             index=["training", "test"])

## 3. Logistic regression

A **pipeline** joins the preparation and the model, like a tidymodels workflow: numbers are scaled, groups become 0/1 columns. Note: scikit-learn's `LogisticRegression` adds a **small penalty** by default (`C=1.0`), so its coefficients differ slightly from R's `glm()`. With 100,000+ people, the predictions barely change.

In [ ]:
logistic = make_pipeline(
    ColumnTransformer([("numbers", StandardScaler(), NUMERIC),
                       ("groups", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
    LogisticRegression(max_iter=1000))
logistic.fit(X_train, y_train)

def score(model, name):
    risk = model.predict_proba(X_test)[:, 1]          # column 1 = the predicted risk of "yes"
    return {"model": name,
            "test_auc": round(roc_auc_score(y_test, risk), 3),
            "brier": round(brier_score_loss(y_test, risk), 4)}

pd.DataFrame([score(logistic, "Logistic regression")])

**🔍 Read the output:** the test AUC and Brier score. Close to 7.1's model 3 (0.743 in our test run)?

## 4. Gradient boosting, tuned

`HistGradientBoostingClassifier` is scikit-learn's fast gradient boosting. It handles groups directly if we turn them into numbers (`OrdinalEncoder`) and tell it which columns are groups (`categorical_features`). `GridSearchCV` tries every combination by 3-fold cross-validation on the training set, like `tune_grid()`. ⏱️ A few minutes.

In [ ]:
boosting = make_pipeline(
    ColumnTransformer([("groups", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL)],
                      remainder="passthrough"),                       # the numbers pass through unchanged
    HistGradientBoostingClassifier(categorical_features=list(range(len(CATEGORICAL))),   # the first 5 columns are groups
                                   max_iter=500, min_samples_leaf=50, random_state=2026))

search = GridSearchCV(boosting,
                      param_grid={"histgradientboostingclassifier__max_depth": [2, 4, 6],
                                  "histgradientboostingclassifier__learning_rate": [0.02, 0.1]},
                      scoring="roc_auc", cv=3, n_jobs=-1)
search.fit(X_train, y_train)

(pd.DataFrame(search.cv_results_)
   .rename(columns={"param_histgradientboostingclassifier__max_depth": "max_depth",
                    "param_histgradientboostingclassifier__learning_rate": "learning_rate"})
   .assign(cv_auc=lambda d: d["mean_test_score"].round(4), std=lambda d: d["std_test_score"].round(4))
   [["max_depth", "learning_rate", "cv_auc", "std"]]
   .sort_values("cv_auc", ascending=False))

**🔍 Read the output:** the 6 combinations, best first, with their cross-validated AUC. Compare with 7.2's Part 4: is the same kind of setting best?

In [ ]:
pd.DataFrame([score(logistic, "Logistic regression"),
              score(search.best_estimator_, "Gradient boosting (best settings)")])

**🔍 Read the output:** the two models on the test set. As in 7.2, expect them to be close.

## 5. Calibration

The same table as in 7.1 and 7.2: tenths of predicted risk, with the share actually diagnosed. Counts of 1 to 20 are hidden, and one more if only one would be.

In [ ]:
risk = search.best_estimator_.predict_proba(X_test)[:, 1]
cal = pd.DataFrame({"risk": risk, "t2d": y_test.to_numpy()})
cal["risk_group"] = pd.qcut(cal["risk"].rank(method="first"), 10, labels=list(range(1, 11)))

table = (cal.groupby("risk_group", observed=True)
            .agg(people=("t2d", "size"), predicted_pct=("risk", "mean"), n=("t2d", "sum"))
            .reset_index())
table["predicted_pct"] = (100 * table["predicted_pct"]).round(2)
table = hide_small_counts(table)
if table["n"].isna().sum() == 1:                       # one hidden count could be worked out from the total
    table.loc[table["n"].idxmin(), "n"] = np.nan
table["n"] = table["n"].astype("Int64")                 # whole numbers, with <NA> for hidden ones
table["observed_pct"] = (100 * table["n"] / table["people"]).round(2)
table.rename(columns={"n": "new_cases"})

**🔍 Read the output:** predicted and observed percentages should be close in each tenth.

## 6. Permutation importance

scikit-learn has it built in: `permutation_importance()` shuffles each column of the test set and measures the drop in AUC, as we did by hand in 7.2. `n_repeats=3` shuffles each column three times, and averages.

In [ ]:
result = permutation_importance(search.best_estimator_, X_test, y_test, scoring="roc_auc",
                                n_repeats=3, random_state=2026, n_jobs=-1)

(pd.DataFrame({"predictor": X_test.columns, "auc_drop": result.importances_mean.round(4)})
   .sort_values("auc_drop", ascending=False))

**🔍 Read the output:** the same ranking as in 7.2? Expect BMI first, then general health and age.

## Your turn

1. Add a **random forest**: `from sklearn.ensemble import RandomForestClassifier`, in a pipeline with `OneHotEncoder` (scikit-learn's forests need numbers). Use `min_samples_leaf=50` and `n_jobs=-1`.
2. Change `random_state=2026` in the split to another number. How much does the test AUC move? That's the noise of one test set.

## Clean up

**Clear outputs** before committing or sharing: **Edit → Clear Outputs of All Cells**, then save.